# Phase 12 — Streamlit Analyst Dashboard

This notebook explains the dashboard architecture and lets you check the FastAPI endpoints from Python. The actual user interface is `dashboard/app.py`. The Streamlit frontend uses the API rather than talking directly to PostgreSQL.

## Dashboard workspaces

1. **Fraud overview** — KPIs, risk distribution, investigation status and activity by day.
2. **Transaction analyzer** — submit `Time`, `V1`–`V28`, and `Amount`; view the model decision and SHAP contribution chart.
3. **High-risk transactions** — triage HIGH/CRITICAL decisions and update investigation status.
4. **Historical predictions** — paginated prediction list with status/risk filters and audit detail.
5. **Monitoring dashboard** — API/model/database readiness, request latency, model versions and workload indicators.

The model score and the 0–100 score are not calibrated real-world probabilities. Flagged predictions are not equivalent to confirmed fraud.

## 1. Start the services

From PowerShell in the repository root, use two terminals. Database commands are needed for history/overview pages; the transaction analyzer can still score when persistence is disabled if the API configuration permits it.

**Terminal 1 — API**

```powershell
.\.venv\Scripts\Activate.ps1
uvicorn api.main:app --reload --host 127.0.0.1 --port 8000
```

**Terminal 2 — Streamlit**

```powershell
.\.venv\Scripts\Activate.ps1
streamlit run dashboard/app.py --server.port 8501
```

Open `http://localhost:8501`. The dashboard's sidebar lets you change the API base URL.

## 2. Check API reachability

This cell uses the same small API client the dashboard uses. If the API isn't running, the client raises a clear connection error.

In [ ]:
from dashboard.api_client import FraudShieldAPIClient, APIClientError

api = FraudShieldAPIClient('http://127.0.0.1:8000')
try:
    health = api.health()
    health
except APIClientError as exc:
    print(f'API unavailable: {exc}')

## 3. Inspect active model metadata

The dashboard displays model version and threshold metadata from `/model-info`; it doesn't duplicate model-loading logic in Streamlit.

In [ ]:
try:
    model_info = api.model_info()
    model_info
except APIClientError as exc:
    print(f'Model metadata unavailable: {exc}')

## 4. Query monitoring summary

`GET /monitoring/summary` is backed by PostgreSQL prediction logs and contains daily volume and latency aggregates. A database-not-configured error is expected before Phase 11's local PostgreSQL setup is complete.

In [ ]:
try:
    summary = api.monitoring_summary()
    print({key: summary.get(key) for key in (
        'total_predictions', 'flagged_predictions', 'high_risk_predictions',
        'mean_latency_ms', 'p50_latency_ms', 'p95_latency_ms', 'database_ready'
    )})
except APIClientError as exc:
    print(f'Monitoring unavailable: {exc}')

## 5. Inspect prediction history

History is paginated, and the API applies risk/status filters before returning records. The user interface lets an analyst change status and records a separate audit event for each update.

In [ ]:
try:
    history = api.list_predictions(limit=10, offset=0)
    print(f"Total matching records: {history.get('total', 0)}")
    history.get('items', [])[:3]
except APIClientError as exc:
    print(f'Prediction history unavailable: {exc}')

## Completion checklist

- [ ] `pip install -r requirements.txt` succeeds and includes Streamlit/Plotly.
- [ ] FastAPI `/health` and `/model-info` return successfully.
- [ ] Streamlit launches at `http://localhost:8501`.
- [ ] Transaction Analyzer submits a complete 30-feature payload and renders the response.
- [ ] With PostgreSQL enabled, a prediction appears in Historical Predictions.
- [ ] Updating a status creates an investigation event.
- [ ] Overview and Monitoring show database-backed aggregates.

**Data limitation:** because raw input features are intentionally not saved with predictions, the monitoring page cannot infer feature drift from history alone. Model quality requires reliable analyst/ground-truth outcomes.